Task 1.3

In [1]:
import numpy as np

In [2]:
# 1) Детектор и ФЭУ
class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.fmt_positions = self.generate_fmts()

    def generate_fmts(self):
        phi = np.random.uniform(0, 2 * np.pi, self.N)
        U = np.random.uniform(-1, 1, self.N)
        sin_theta = np.sqrt(1 - U**2)

        x = self.R * sin_theta * np.cos(phi)
        y = self.R * sin_theta * np.sin(phi)
        z = self.R * U

        return np.column_stack([x, y, z])

    def find_fmt(self, point):
        d = np.sqrt(np.sum((self.fmt_positions - point)**2, axis=1))
        i = np.argmin(d)

        if d[i] <= self.r:
            return int(i)
        return -1

    # 2) Пересечение фотона со сферой
    def get_intersection(self, pos, direction):
        b = np.dot(pos, direction)
        c = np.dot(pos, pos) - self.R**2
        L = -b + np.sqrt(b**2 - c)
        return pos + L * direction

    def generate_fe(self):
        return np.random.poisson(self.a)

In [3]:
# 3) Источник и направления фотонов
class Source:
    def __init__(self, Y=10000, pos=(0.0, 0.0, 0.0)):
        self.Y = Y
        self.pos = np.array(pos, dtype=float) # источник может быть не в центре

    def generate_directions(self, E):
        N = int(self.Y * E)

        phi = np.random.uniform(0, 2 * np.pi, N)
        U = np.random.uniform(-1, 1, N)
        sin_theta = np.sqrt(1 - U**2)

        x = sin_theta * np.cos(phi)
        y = sin_theta * np.sin(phi)
        z = U

        return np.column_stack([x, y, z])

In [4]:
# 4) Параметры
R = 2.0
N = 40
r = 0.1
a = 0.5
Y = 10000
E = 1.0

det = Detector(R, N, r, a)
source = Source(Y, pos=(0.0, 0.0, 0.0))

In [5]:
# 5) Генерация направлений фотонов
directions = source.generate_directions(E)
N_photons = len(directions)

In [6]:
# 6) Попадания на ФЭУ и регистрация фотоэлектронов
Q_fmt = np.zeros(det.N, dtype=int)
N_hit = 0

for n in directions:
    point = det.get_intersection(source.pos, n)
    i = det.find_fmt(point)

    if i != -1:
        N_hit += 1
        Q_fmt[i] += det.generate_fe()

In [7]:
# 7) Полный заряд и оценка
Q_total = np.sum(Q_fmt)
Q_ozhid = det.a * source.Y * E * (det.N * det.r**2) / det.R**2

print(f"Всего излучено фотонов: {N_photons}")
print(f"Попало фотонов: {N_hit}")
print(f"Полный собранный заряд Q: {Q_total}")
print(f"Оценка: {Q_ozhid:.2f}")

Всего излучено фотонов: 10000
Попало фотонов: 273
Полный собранный заряд Q: 136
Оценка: 500.00
